In [ ]:
# Import des bibliothèques
import sys
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Ajouter le chemin vers le module src
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.vision import Camera, CAMERA_MATRIX, CAMERA_COEFFICIENTS

# Test de Calibration de la Caméra

Ce notebook teste les paramètres de calibration de la caméra en affichant une image avant et après correction de la distorsion.

## 1. Affichage des paramètres de calibration

In [ ]:
# Afficher les paramètres de calibration actuels
print("="*70)
print("PARAMÈTRES DE CALIBRATION DE LA CAMÉRA")
print("="*70)

print("\nMatrice de la caméra (paramètres intrinsèques):")
print(CAMERA_MATRIX)
print(f"\n  fx = {CAMERA_MATRIX[0, 0]:.2f}  (distance focale en x)")
print(f"  fy = {CAMERA_MATRIX[1, 1]:.2f}  (distance focale en y)")
print(f"  cx = {CAMERA_MATRIX[0, 2]:.2f}  (centre optique en x)")
print(f"  cy = {CAMERA_MATRIX[1, 2]:.2f}  (centre optique en y)")

print("\nCoefficients de distorsion [k1, k2, p1, p2, k3]:")
print(CAMERA_COEFFICIENTS)
print(f"  k1 = {CAMERA_COEFFICIENTS[0]:.6f}  (distorsion radiale)")
print(f"  k2 = {CAMERA_COEFFICIENTS[1]:.6f}  (distorsion radiale)")
print(f"  p1 = {CAMERA_COEFFICIENTS[2]:.6f}  (distorsion tangentielle)")
print(f"  p2 = {CAMERA_COEFFICIENTS[3]:.6f}  (distorsion tangentielle)")
print(f"  k3 = {CAMERA_COEFFICIENTS[4]:.6f}  (distorsion radiale)")
print("="*70)

## 2. Capture d'une image de test

In [ ]:
# Initialiser la caméra
print("Initialisation de la caméra...")
cam = Camera(cam_id=0)

# Attendre que la caméra s'ajuste
import time
print("Attente de stabilisation de la caméra (2 secondes)...")
time.sleep(2)

# Capturer une image
ret, frame = cam.get_frame()

if ret:
    print(f"✓ Image capturée avec succès! Dimensions: {frame.shape[1]}x{frame.shape[0]}")
    original_frame = frame.copy()
else:
    print("✗ Erreur lors de la capture de l'image")
    
# Arrêter la caméra
cam.stop()

## 3. Application de la correction de distorsion

In [ ]:
# Obtenir les dimensions de l'image
h, w = original_frame.shape[:2]

# Calculer la nouvelle matrice de caméra optimale
print("\nCalcul de la matrice de caméra optimale...")
newcameramtx, roi = cv2.getOptimalNewCameraMatrix(
    CAMERA_MATRIX, 
    CAMERA_COEFFICIENTS, 
    (w, h), 
    1,  # alpha=1 pour garder tous les pixels
    (w, h)
)

print(f"✓ Nouvelle matrice calculée")
print(f"  ROI (Region of Interest): x={roi[0]}, y={roi[1]}, w={roi[2]}, h={roi[3]}")

# Appliquer la correction de distorsion
print("\nApplication de la correction de distorsion...")
undistorted_frame = cv2.undistort(
    original_frame, 
    CAMERA_MATRIX, 
    CAMERA_COEFFICIENTS, 
    None, 
    newcameramtx
)

# Rogner l'image selon la ROI
x, y, w_roi, h_roi = roi
undistorted_cropped = undistorted_frame[y:y+h_roi, x:x+w_roi]

print(f"✓ Correction appliquée avec succès!")
print(f"  Image originale: {original_frame.shape[1]}x{original_frame.shape[0]}")
print(f"  Image corrigée (rognée): {undistorted_cropped.shape[1]}x{undistorted_cropped.shape[0]}")

## 4. Comparaison visuelle : Avant / Après correction

In [ ]:
# Convertir BGR vers RGB pour matplotlib
original_rgb = cv2.cvtColor(original_frame, cv2.COLOR_BGR2RGB)
undistorted_rgb = cv2.cvtColor(undistorted_cropped, cv2.COLOR_BGR2RGB)

# Créer une figure avec deux sous-graphiques
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Image AVANT correction
axes[0].imshow(original_rgb)
axes[0].set_title('AVANT Correction\n(Image Distordue)', fontsize=14, fontweight='bold')
axes[0].axis('off')
axes[0].text(0.5, -0.05, f'Dimensions: {original_frame.shape[1]}x{original_frame.shape[0]} px', 
             ha='center', transform=axes[0].transAxes, fontsize=10)

# Image APRÈS correction
axes[1].imshow(undistorted_rgb)
axes[1].set_title('APRÈS Correction\n(Image Sans Distorsion)', fontsize=14, fontweight='bold', color='green')
axes[1].axis('off')
axes[1].text(0.5, -0.05, f'Dimensions: {undistorted_cropped.shape[1]}x{undistorted_cropped.shape[0]} px', 
             ha='center', transform=axes[1].transAxes, fontsize=10)

plt.tight_layout()
plt.suptitle('Test de Calibration - Correction de la Distorsion de la Caméra', 
             fontsize=16, fontweight='bold', y=1.02)
plt.show()

print("\n💡 Conseil: Observez les bords de l'image et les lignes droites.")
print("   Les lignes courbes dans l'image originale devraient être droites après correction.")

## 5. Analyse des différences (optionnel)

Calcul de la différence pixel par pixel entre les deux images pour visualiser l'effet de la correction.

In [ ]:
# Redimensionner l'image originale pour correspondre à l'image corrigée rognée
original_resized = cv2.resize(original_frame, (undistorted_cropped.shape[1], undistorted_cropped.shape[0]))

# Calculer la différence absolue
difference = cv2.absdiff(original_resized, undistorted_cropped)

# Convertir en niveaux de gris pour mieux voir les différences
difference_gray = cv2.cvtColor(difference, cv2.COLOR_BGR2GRAY)

# Créer une carte de chaleur
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Afficher l'image de différence
difference_rgb = cv2.cvtColor(difference, cv2.COLOR_BGR2RGB)
axes[0].imshow(difference_rgb)
axes[0].set_title('Différence Absolue (RGB)', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Afficher la carte de chaleur de l'intensité des différences
im = axes[1].imshow(difference_gray, cmap='hot')
axes[1].set_title('Carte de Chaleur des Différences\n(Rouge = Forte correction)', 
                  fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.suptitle('Analyse des Différences - Impact de la Correction de Distorsion', 
             fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Calculer quelques statistiques
mean_diff = np.mean(difference_gray)
max_diff = np.max(difference_gray)
pixels_changed = np.sum(difference_gray > 5)  # Pixels avec différence > 5
total_pixels = difference_gray.shape[0] * difference_gray.shape[1]
percent_changed = (pixels_changed / total_pixels) * 100

print("\n" + "="*70)
print("STATISTIQUES DE CORRECTION")
print("="*70)
print(f"Différence moyenne:          {mean_diff:.2f} (sur 255)")
print(f"Différence maximale:         {max_diff:.2f} (sur 255)")
print(f"Pixels significativement modifiés: {pixels_changed:,} ({percent_changed:.1f}%)")
print("="*70)